# Part 6 - Validation, Traceability and Reflexion

## Goal

From the planning document:

- *"A Validation Agent checks completeness, consistency, and traceability."*
- *"The validator maps every requirement to acceptance criteria, design sections, and tests."*
- *"Validation failures return the package to the responsible agent."*
- *"Reflexion helps diagnose failed validation ... revise the output, and retry safely without bypassing human approval."*

The Validation Agent has two layers:

1. **Deterministic checks** in Python: a traceability matrix from requirement IDs to acceptance criteria,
   design sections and tests, plus references to IDs that were never approved. Code is more reliable than an
   LLM for these checks.
2. **An LLM consistency review**: it compares the artefacts with the immutable facts and with each other
   (wrong values, contradictions, behaviour nobody asked for).

Each issue names the **responsible agent**. A LangGraph loop sends the artefact back to that agent with the
feedback and a **reflexion memory** of earlier failures, then validates again, up to a maximum number of revisions.

## Setup

Same environment as the course notebooks: a local **Ollama** server running **`gemma4:e4b`** for the agents and **`nomic-embed-text`** for embeddings. The cells work on Google Colab (Ollama is installed automatically) and locally (install Ollama from ollama.com first).

This notebook only needs the chat model.

**Chat model:** set `LLM_PROVIDER=openai` in `.env` to run the agents on **GPT Luna** (`OPENAI_MODEL`, default `gpt-6-luna`, key in `OPEN_AI_KEY`); Ollama is then not needed for this notebook. `LLM_PROVIDER=ollama` (the default) keeps the local `gemma4:e4b`.

In [ ]:
%pip install -q -U langchain langchain-core langchain-ollama langchain-openai python-dotenv langchain-text-splitters langgraph langchain-mcp-adapters mcp python-docx requests numpy langsmith

In [ ]:
import os
import platform
import shutil
import subprocess
import time

import requests
from dotenv import load_dotenv

load_dotenv()  # reads .env next to the notebook (OPEN_AI_KEY, LLM_PROVIDER, ADO_*, ...) when it exists

# Chat model: "openai" runs the agents on GPT Luna (key in OPEN_AI_KEY), "ollama" on the local model.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()
if LLM_PROVIDER not in ("openai", "ollama"):
    raise ValueError(f"LLM_PROVIDER must be 'openai' or 'ollama', not {LLM_PROVIDER!r}")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e4b")
LLM_MODEL = OPENAI_MODEL if LLM_PROVIDER == "openai" else OLLAMA_MODEL
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_URL = "http://localhost:11434"
NEEDS_OLLAMA = LLM_PROVIDER == "ollama"  # this notebook only needs the chat model


def ollama_is_running() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).ok
    except requests.RequestException:
        return False


# 1. Install Ollama (Colab / Linux) and start the server in the background
if NEEDS_OLLAMA:
    if shutil.which("ollama") is None:
        if platform.system() == "Linux":
            print("Installing Ollama...")
            !sudo apt-get install -y -qq zstd
            !curl -fsSL https://ollama.com/install.sh | sh
        else:
            print("Install Ollama from https://ollama.com/download, then re-run this cell.")

    if not ollama_is_running():
        print("Starting Ollama server...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)  # Wait for server to wake up

    print("Ollama running:", ollama_is_running())
print(f"Chat model: {LLM_PROVIDER} / {LLM_MODEL}")

In [ ]:
# 2. Pull the chat model and the embedding model (only when the agents run on Ollama)
if NEEDS_OLLAMA:
    !ollama pull {LLM_MODEL}
    !ollama pull {EMBED_MODEL}
    !ollama list

### LangSmith tracing

Every agent, LLM call, MCP tool call and LangGraph node is traced in **LangSmith**, so you can open each step
and see its **prompt, output, duration and tokens**. Paste your API key when asked (or store it as the Colab
secret `LANGSMITH_API_KEY`); press Enter to run without tracing.

In [ ]:
import getpass
import os

import langsmith.utils

# LangSmith records every agent, LLM call (full prompt and answer), MCP tool call and LangGraph node,
# with its duration and token usage. Free API key: https://smith.langchain.com -> Settings -> API Keys.
# Note: traces contain the prompts, so customer emails are sent to LangSmith. Use test data or your own workspace.
USE_LANGSMITH = True
LANGSMITH_PROJECT = "fo-requirements-to-delivery"


def read_langsmith_key() -> str:
    if os.getenv("LANGSMITH_API_KEY"):
        return os.environ["LANGSMITH_API_KEY"]
    try:
        from google.colab import userdata  # Colab: add a secret named LANGSMITH_API_KEY

        return userdata.get("LANGSMITH_API_KEY")
    except Exception:
        pass
    try:
        return getpass.getpass("LangSmith API key (press Enter to run without tracing): ").strip()
    except Exception:  # no interactive input available
        return ""


api_key = read_langsmith_key() if USE_LANGSMITH else ""
if api_key:
    os.environ.update(LANGSMITH_API_KEY=api_key, LANGSMITH_TRACING="true", LANGSMITH_PROJECT=LANGSMITH_PROJECT)
    print(f"LangSmith tracing ON -> project '{LANGSMITH_PROJECT}' on https://smith.langchain.com")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")
langsmith.utils.get_env_var.cache_clear()  # LangSmith caches environment variables

In [ ]:
import getpass
import os


def make_llm():
    """The chat model selected by LLM_PROVIDER: GPT Luna on OpenAI, or the local Ollama model."""
    if LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI

        if not (os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY")):
            os.environ["OPEN_AI_KEY"] = getpass.getpass("OpenAI API key: ").strip()
        # GPT Luna only accepts its default temperature, so none is set.
        return ChatOpenAI(model=LLM_MODEL, api_key=os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY"))

    from langchain_ollama import ChatOllama

    return ChatOllama(model=LLM_MODEL, temperature=0)


llm = make_llm()

In [ ]:
import json

from langchain_core.messages import HumanMessage, SystemMessage
from langsmith import traceable  # names agent steps in LangSmith (does nothing when tracing is off)


def ask_structured(schema, system_prompt: str, user_prompt: str, retries: int = 2):
    """Call the LLM and force the answer into a Pydantic schema. On failure, retry with the error message."""
    # The run name shows up in LangSmith, e.g. "DeliveryPlan" or "GapAnalysis".
    structured_llm = llm.with_structured_output(schema, method="json_schema").with_config(run_name=schema.__name__)
    last_error = None

    for attempt in range(retries + 1):
        try:
            result = structured_llm.invoke(
                [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
            )
            if result is None:
                raise ValueError("The model returned an empty answer.")
            return result
        except Exception as error:
            last_error = error
            print(f"[retry {attempt + 1}] invalid structured output: {error}")
            user_prompt += (
                f"\n\nYour previous answer was invalid ({error}). "
                "Return JSON that matches the schema exactly."
            )

    raise last_error


def ask_text(system_prompt: str, user_prompt: str) -> str:
    return llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    ).text  # plain string for both providers (OpenAI returns content blocks)


def show(title: str, obj) -> None:
    """Pretty-print a heading and a Pydantic object, dict, list or string."""
    print(f"\n========== {title} ==========\n")
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    print(obj if isinstance(obj, str) else json.dumps(obj, indent=2, ensure_ascii=False))
import operator
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Requirement(BaseModel):
    req_id: str = Field(description="Stable ID such as REQ-001")
    statement: str = Field(description="One clear, testable requirement sentence")
    source_text: str = Field(description="The customer's words this requirement comes from")
    actor: str = Field(description="Who performs or benefits from the requirement")
    business_rules: list[str] = Field(description="Rules, thresholds or conditions that are actually stated")
    exceptions: list[str] = Field(description="Exceptions or edge cases that are actually stated")
    expected_outcome: str = Field(description="What should happen when the requirement is met")


class RequirementExtraction(BaseModel):
    summary: str = Field(description="Two-sentence summary of the request")
    actors: list[str]
    in_scope: list[str]
    out_of_scope: list[str]
    security_constraints: list[str] = Field(description="Security, approval or segregation-of-duties constraints")
    requirements: list[Requirement]


class Classification(BaseModel):
    module: Literal["Finance", "Supply Chain", "Finance and Supply Chain", "Out of scope"]
    sub_areas: list[str] = Field(description="F&O areas such as Accounts payable, Procurement, Inventory")
    work_type: Literal["Functional", "Technical", "Mixed"]
    design_documents: Literal["FDD", "TDD", "FDD and TDD"]
    risk_level: Literal["Low", "Medium", "High"]
    rationale: str


class Gap(BaseModel):
    gap_id: str = Field(description="Stable ID such as GAP-01")
    req_id: str = Field(description="The requirement this gap belongs to")
    missing_information: str
    why_it_matters: str
    clarification_question: str = Field(description="One clear question the customer can answer")
    assumption_if_unanswered: str


class GapAnalysis(BaseModel):
    gaps: list[Gap]
class AcceptanceCriterion(BaseModel):
    ac_id: str = Field(description="ID such as AC-01")
    req_ids: list[str]
    given: str
    when: str
    then: str


class TaskItem(BaseModel):
    task_id: str = Field(description="ID such as T-01")
    kind: Literal["Functional", "Technical"]
    title: str = Field(description="Starts with FUNC: or TECH:")
    description: str
    req_ids: list[str]
    estimate_hours: float


class UserStory(BaseModel):
    story_id: str = Field(description="ID such as US-01")
    title: str = Field(description="Format: [Module] short goal")
    as_a: str
    i_want: str
    so_that: str
    req_ids: list[str]
    acceptance_criteria: list[AcceptanceCriterion]
    tasks: list[TaskItem]


class DeliveryPlan(BaseModel):
    user_stories: list[UserStory]
class DesignSection(BaseModel):
    section_id: str = Field(description="ID such as FDD-1 or TDD-1")
    heading: str
    content: str
    req_ids: list[str] = Field(description="Requirement IDs this section covers")


class DesignDocument(BaseModel):
    doc_type: Literal["FDD", "TDD"]
    title: str
    overview: str
    sections: list[DesignSection]
    open_points: list[str]


class TestCase(BaseModel):
    test_id: str = Field(description="ID such as TC-01")
    test_type: Literal["Functional", "Technical", "Integration", "Regression"]
    title: str
    req_ids: list[str]
    preconditions: str
    steps: list[str]
    expected_result: str


class TestSuite(BaseModel):
    test_cases: list[TestCase]
AgentName = Literal["planning", "functional_design", "technical_design", "tests"]


class ValidationIssue(BaseModel):
    severity: Literal["Blocker", "Major", "Minor"]
    responsible_agent: AgentName
    req_ids: list[str]
    description: str
    fix_instruction: str


class ConsistencyReview(BaseModel):
    issues: list[ValidationIssue]

## Inputs

The approved specification and the plan are correct. The FDD, TDD and tests below contain **three deliberate
defects**, so we can see whether the validator catches them and whether the loop fixes them.

In [ ]:
REQUEST_ID = "CR-2026-001"

CUSTOMER_REQUEST = """
Subject: Vendor invoices approval + blocking vendors

Hi team,

Going forward we need vendor invoices above a certain amount to be approved by the
finance manager before they can be posted. Right now anyone in AP can post them and
the auditors flagged it.

Also, vendors that don't have bank details should be blocked so we don't pay them by
mistake. Some of the old vendors are like this too.

Can we have this for all our companies? We would like it live before the quarter close.

Thanks,
Rana Haddad
AP Team Lead, Cedar Retail Group
"""
REQUEST_ID = "CR-2026-001"
DEADLINE = "2026-12-15"

CLASSIFICATION = {
    "module": "Finance",
    "sub_areas": ["Accounts payable"],
    "work_type": "Mixed",
    "design_documents": "FDD and TDD",
    "risk_level": "Medium",
    "rationale": "Approval workflow configuration (functional) plus a vendor payment-hold batch job and validation (technical).",
}

APPROVED_REQUIREMENTS = [
    {"req_id": "REQ-001", "actor": "Finance Manager",
     "statement": "Vendor invoices with a total including tax greater than 10,000 USD (accounting currency) must be approved by the Finance Manager of the legal entity before they can be posted.",
     "business_rules": ["Threshold 10,000 USD incl. tax, accounting currency", "Legal entities USMF and DEMF"],
     "exceptions": [], "expected_outcome": "Invoices above the threshold cannot be posted until approved."},
    {"req_id": "REQ-002", "actor": "AP Clerk",
     "statement": "Vendor invoices of 10,000 USD or less can be posted by AP clerks without approval.",
     "business_rules": ["Threshold 10,000 USD incl. tax"], "exceptions": [],
     "expected_outcome": "Current posting process is unchanged for small invoices."},
    {"req_id": "REQ-003", "actor": "CFO",
     "statement": "If the Finance Manager does not act within 2 business days, the approval escalates to the CFO.",
     "business_rules": ["Escalation after 2 business days"], "exceptions": [],
     "expected_outcome": "Pending approvals do not block posting indefinitely."},
    {"req_id": "REQ-004", "actor": "AP Team",
     "statement": "Existing and new vendors without at least one active bank account are placed on payment hold, which blocks payment generation but still allows invoice registration.",
     "business_rules": ["On hold = Payment", "Applies to existing and new vendors"], "exceptions": [],
     "expected_outcome": "No payment can be generated for vendors without bank details."},
    {"req_id": "REQ-005", "actor": "Auditor",
     "statement": "The approval history (approver, decision, date and time) of each vendor invoice must be visible to auditors.",
     "business_rules": [], "exceptions": [], "expected_outcome": "Auditors can prove who approved each invoice and when."},
    {"req_id": "REQ-006", "actor": "AP Clerk",
     "statement": "AP clerks must not be able to approve vendor invoices they entered themselves.",
     "business_rules": ["Segregation of duties"], "exceptions": [],
     "expected_outcome": "Self-approval is technically prevented."},
]

LOCKED_FACTS = [
    "Request ID: CR-2026-001",
    "Legal entities in scope: USMF and DEMF only",
    "Approval threshold: vendor invoice total including tax greater than 10,000 USD in accounting currency",
    "Approver: Finance Manager of the legal entity; escalate to the CFO after 2 business days",
    "Blocked vendor means On hold = Payment (invoices can still be registered)",
    "Security: AP clerks must not approve invoices they entered",
    "Go-live deadline: 2026-12-15",
    "Cited evidence: MS-WF-01, MS-AP-01, MS-SEC-01, DES-2025-014, DES-2025-031",
]

EVIDENCE = [
    {"doc_id": "MS-WF-01", "title": "Workflow and approvals", "source_type": "microsoft_guidance", "module": "Finance",
     "snippet": "Approval steps can have conditions, such as the invoice total being greater than an amount... time limits and escalation paths... workflow history records who approved and when."},
    {"doc_id": "MS-AP-01", "title": "Vendor invoices and holds", "source_type": "microsoft_guidance", "module": "Finance",
     "snippet": "On hold = Payment blocks payment generation while invoices can still be registered."},
    {"doc_id": "DES-2025-014", "title": "Approved design: purchase invoice approval by amount", "source_type": "approved_design", "module": "Finance",
     "snippet": "Workflow condition on invoice total (accounting currency) > threshold; SoD rule; threshold in a CUS parameter."},
    {"doc_id": "DES-2025-031", "title": "Approved design: automatic hold for vendors with incomplete master data", "source_type": "approved_design", "module": "Finance",
     "snippet": "Recurring SysOperation batch sets On hold = Payment; CoC validation on vendor insert/update; CUS log table."},
]
# Output of the Planning Agent for CR-2026-001 (see notebook 04)
SAMPLE_PLAN = {"user_stories": [
    {"story_id": "US-01", "title": "[Finance] Approve large vendor invoices before posting",
     "as_a": "Finance Manager", "i_want": "to approve vendor invoices above 10,000 USD before they are posted",
     "so_that": "large payments are controlled and auditable",
     "req_ids": ["REQ-001", "REQ-002", "REQ-003", "REQ-005", "REQ-006"],
     "acceptance_criteria": [
         {"ac_id": "AC-01", "req_ids": ["REQ-001"], "given": "a vendor invoice in USMF with a total of 10,000.01 USD including tax",
          "when": "an AP clerk submits it", "then": "it cannot be posted until the Finance Manager approves it"},
         {"ac_id": "AC-02", "req_ids": ["REQ-002"], "given": "a vendor invoice of 10,000 USD or less",
          "when": "an AP clerk posts it", "then": "it posts without approval"},
         {"ac_id": "AC-03", "req_ids": ["REQ-003"], "given": "an approval pending for more than 2 business days",
          "when": "the escalation time limit is reached", "then": "the approval is escalated to the CFO"},
         {"ac_id": "AC-04", "req_ids": ["REQ-005"], "given": "an approved vendor invoice",
          "when": "an auditor opens the workflow history", "then": "the approver, decision, date and time are shown"},
         {"ac_id": "AC-05", "req_ids": ["REQ-006"], "given": "an AP clerk who entered an invoice",
          "when": "the same clerk tries to approve it", "then": "the approval is prevented"},
     ],
     "tasks": [
         {"task_id": "T-01", "kind": "Functional", "title": "FUNC: Configure the vendor invoice workflow with threshold and escalation in USMF and DEMF",
          "description": "Workflow condition on invoice total incl. tax > 10,000 USD, Finance Manager step, CFO escalation after 2 business days.",
          "req_ids": ["REQ-001", "REQ-002", "REQ-003"], "estimate_hours": 12},
         {"task_id": "T-02", "kind": "Functional", "title": "FUNC: Write the FDD for invoice approval and vendor payment hold",
          "description": "Functional design document.", "req_ids": ["REQ-001", "REQ-002", "REQ-003", "REQ-004", "REQ-005", "REQ-006"], "estimate_hours": 8},
         {"task_id": "T-03", "kind": "Technical", "title": "TECH: Add the CUS threshold parameter, privileges and SoD rule",
          "description": "Parameter table field, security privilege/duty extensions, SoD rule between entering and approving invoices.",
          "req_ids": ["REQ-001", "REQ-006"], "estimate_hours": 10},
         {"task_id": "T-04", "kind": "Functional", "title": "FUNC: Give auditors read access to the workflow history",
          "description": "Role setup for auditors.", "req_ids": ["REQ-005"], "estimate_hours": 3},
     ]},
    {"story_id": "US-02", "title": "[Finance] Hold payments for vendors without bank details",
     "as_a": "AP Team Lead", "i_want": "vendors without an active bank account to be on payment hold",
     "so_that": "we never pay a vendor by mistake", "req_ids": ["REQ-004"],
     "acceptance_criteria": [
         {"ac_id": "AC-06", "req_ids": ["REQ-004"], "given": "an existing vendor without an active bank account",
          "when": "the vendor check batch job runs", "then": "On hold is set to Payment and invoices can still be registered"},
         {"ac_id": "AC-07", "req_ids": ["REQ-004"], "given": "a new vendor saved without a bank account",
          "when": "the vendor is saved", "then": "the vendor is placed on payment hold"},
     ],
     "tasks": [
         {"task_id": "T-05", "kind": "Technical", "title": "TECH: Build the CUSVendBankAccountCheck SysOperation batch job",
          "description": "Sets and releases On hold = Payment per legal entity, idempotent, logs every change.",
          "req_ids": ["REQ-004"], "estimate_hours": 16},
         {"task_id": "T-06", "kind": "Technical", "title": "TECH: Chain of Command validation on vendor insert/update and CUS log table",
          "description": "CoC on VendTable insert/update, CUSVendHoldLog table.", "req_ids": ["REQ-004"], "estimate_hours": 10},
         {"task_id": "T-07", "kind": "Technical", "title": "TECH: Write the TDD",
          "description": "Technical design document.", "req_ids": ["REQ-001", "REQ-003", "REQ-004", "REQ-006"], "estimate_hours": 6},
         {"task_id": "T-08", "kind": "Functional", "title": "FUNC: Schedule the batch job in USMF and DEMF and train the AP team",
          "description": "Batch recurrence and user training.", "req_ids": ["REQ-004"], "estimate_hours": 4},
     ]},
]}

In [ ]:
# Artefacts produced by the design and test agents, with three deliberate defects:
#   1. FDD-1 uses a threshold of 5,000 USD (the approved fact is 10,000 USD)
#   2. no test case covers REQ-005 (audit history)
#   3. TC-06 references REQ-009, which is not an approved requirement
DRAFT_FDD = {
    "doc_type": "FDD", "title": "FDD - Vendor invoice approval and vendor payment hold",
    "overview": "Adds an approval step for large vendor invoices and a payment hold for vendors without bank details in USMF and DEMF.",
    "sections": [
        {"section_id": "FDD-1", "heading": "Business context and to-be process", "req_ids": ["REQ-001", "REQ-002"],
         "content": "Vendor invoices above 5,000 USD must be approved by the Finance Manager before posting. Smaller invoices are posted by AP clerks as today [MS-WF-01]."},
        {"section_id": "FDD-2", "heading": "Workflow configuration", "req_ids": ["REQ-001", "REQ-003"],
         "content": "Vendor invoice workflow per legal entity with a condition on the invoice total; approval step assigned to the Finance Manager role; escalation to the CFO after 2 business days [DES-2025-014]."},
        {"section_id": "FDD-3", "heading": "Vendor payment hold", "req_ids": ["REQ-004"],
         "content": "Vendors without an active bank account are set to On hold = Payment; invoices can still be registered [MS-AP-01]."},
        {"section_id": "FDD-4", "heading": "Security and audit", "req_ids": ["REQ-005", "REQ-006"],
         "content": "Auditors get read access to the workflow history. An SoD rule prevents AP clerks from approving invoices they entered [MS-SEC-01]."},
    ],
    "open_points": [],
}

DRAFT_TDD = {
    "doc_type": "TDD", "title": "TDD - Vendor invoice approval and vendor payment hold",
    "overview": "Extensions in the CUSExtensions model.",
    "sections": [
        {"section_id": "TDD-1", "heading": "Parameters and security objects", "req_ids": ["REQ-001", "REQ-006"],
         "content": "CUSApprovalThreshold field on the AP parameters table extension (default 10,000 USD); privilege and duty extensions; SoD rule."},
        {"section_id": "TDD-2", "heading": "Vendor bank account check batch", "req_ids": ["REQ-004"],
         "content": "SysOperation classes CUSVendBankAccountCheckContract/Controller/Service; sets and releases On hold = Payment per legal entity; idempotent [DES-2025-031]."},
        {"section_id": "TDD-3", "heading": "Logging and audit", "req_ids": ["REQ-004", "REQ-005"],
         "content": "CUSVendHoldLog table for hold changes; invoice approvals use the standard workflow history."},
    ],
    "open_points": [],
}

DRAFT_TESTS = {"test_cases": [
    {"test_id": "TC-01", "test_type": "Functional", "title": "Invoice just above the threshold needs approval", "req_ids": ["REQ-001"],
     "preconditions": "Workflow active in USMF", "steps": ["Create an invoice of 10,000.01 USD incl. tax", "Submit", "Try to post"],
     "expected_result": "Posting is blocked until the Finance Manager approves."},
    {"test_id": "TC-02", "test_type": "Functional", "title": "Invoice at the threshold posts without approval", "req_ids": ["REQ-002"],
     "preconditions": "Workflow active in USMF", "steps": ["Create an invoice of exactly 10,000 USD", "Post"],
     "expected_result": "The invoice posts without approval."},
    {"test_id": "TC-03", "test_type": "Integration", "title": "Escalation to the CFO", "req_ids": ["REQ-003"],
     "preconditions": "Pending approval", "steps": ["Leave the approval pending for 2 business days"],
     "expected_result": "The approval is escalated to the CFO."},
    {"test_id": "TC-04", "test_type": "Technical", "title": "Batch sets payment hold", "req_ids": ["REQ-004"],
     "preconditions": "Vendor without bank account", "steps": ["Run CUSVendBankAccountCheck"],
     "expected_result": "On hold = Payment; invoices can still be registered."},
    {"test_id": "TC-05", "test_type": "Functional", "title": "Clerk cannot approve own invoice", "req_ids": ["REQ-006"],
     "preconditions": "Clerk entered the invoice", "steps": ["Clerk opens the approval", "Clerk tries to approve"],
     "expected_result": "The approval is prevented."},
    {"test_id": "TC-06", "test_type": "Regression", "title": "Foreign-currency invoice posting", "req_ids": ["REQ-009"],
     "preconditions": "EUR vendor", "steps": ["Post a EUR invoice below the threshold"],
     "expected_result": "The invoice posts as before."},
]}

In [ ]:
# Company standards (the same text that is stored in the knowledge base, notebook 02)
STANDARDS = {
    'STD-DEV-01': 'All custom objects use the prefix CUS and are placed in the model CUSExtensions. Customizations must use extensions and Chain of Command; overlayering and copying standard code are not allowed. Configurable values such as thresholds, days or amounts are stored in a parameter table exposed on a parameters form, never hard-coded. Every new menu item needs a privilege and a duty, delivered as extensions of existing roles. Business logic must have SysTest unit tests, and each change goes through pull-request code review in Azure DevOps.',
    'STD-DOC-01': 'A Functional Design Document (FDD) is required when a business process or configuration changes. It contains business context, as-is and to-be process, configuration and parameters, roles and security, exceptions, reports and data, assumptions and open points. A Technical Design Document (TDD) is required when code, integrations or reports are created. It contains solution overview, object list (extensions, classes, tables, security objects), logic, data model changes, error handling and logging, performance and batch considerations, and deployment steps. Mixed changes need both documents. Every section references the requirement IDs it covers.',
    'STD-ADO-01': "Each approved requirement is delivered through at least one User Story. Story titles use the format '[Module] short goal', and the description uses 'As a <role>, I want <goal> so that <benefit>'. Acceptance criteria are written as Given/When/Then and reference requirement IDs. Implementation work is split into child Tasks: functional tasks are prefixed 'FUNC:' (configuration, FDD, training) and technical tasks are prefixed 'TECH:' (development, TDD, unit tests). Tasks carry an estimate in hours in Remaining Work. Stories are tagged with the module, work type and request ID. Assignees are proposed and then confirmed by a person; tools must not assign work automatically.",
    'STD-TEST-01': 'Each requirement is covered by at least one test case, and test cases reference requirement IDs. Test types: Functional (business scenario in the UI), Technical (unit or component behaviour, such as a SysTest), Integration (interaction with workflow, batch, data entities or external systems) and Regression (standard processes that must still work). Threshold rules need boundary tests at, just below and just above the limit. Security requirements need a negative test proving that an unauthorized user is blocked.',
    'STD-COMM-01': 'Clarification emails are short, polite and written in business language without technical jargon. Questions are numbered, one topic per question, with at most eight questions per email. Each email restates the request in one sentence, asks for a reply date, and never promises a delivery date or solution before the requirement is approved. Emails are saved as drafts and sent only after a person reviews them.',
}

In [ ]:
def handoff_package(state: dict) -> str:
    """The structured package passed at every handoff (Coordination section of the planning document)."""
    package = {
        "request_id": state.get("request_id"),
        "source_text": state.get("source_text", ""),
        "requirements": state.get("requirements", []),
        "classification": state.get("classification"),
        "confirmed_answers": state.get("confirmed_answers", {}),
        "assumptions": state.get("assumptions", []),
        "retrieved_evidence": [
            {**e, "snippet": e["snippet"][:400]} for e in state.get("evidence", [])
        ],
        "approval_status": state.get("approval_status", {}),
        "immutable_facts": state.get("locked_facts", []),
    }
    return json.dumps(package, indent=2, ensure_ascii=False)
PLANNING_SYSTEM = """
You are the Planning Agent of a Dynamics 365 F&O delivery team.
Create Azure DevOps user stories from the approved specification.

Rules:
- Every requirement ID must be covered by at least one user story and at least one acceptance criterion.
- Story titles use the format "[Module] short goal". Acceptance criteria use Given/When/Then, IDs AC-01, AC-02, ...
- Split the work into tasks, IDs T-01, T-02, ...: functional tasks start with "FUNC:" (configuration,
  FDD, training) and technical tasks start with "TECH:" (X++ extensions, security objects, batch jobs,
  TDD, unit tests). Give each task an estimate in hours.
- The immutable facts are final: use their exact values and never change them.
"""

FDD_SYSTEM = """
You are the Functional Design Agent of a Dynamics 365 F&O delivery team.
Write the Functional Design Document (FDD) for the approved specification.

- Follow the company design document standard: business context, as-is and to-be process,
  configuration and parameters, roles and security, exceptions, reports and audit, assumptions.
- Section IDs FDD-1, FDD-2, ... Each section lists the requirement IDs it covers, and every requirement
  must appear in at least one section.
- Use only the immutable facts and confirmed answers. Cite evidence IDs in the text, e.g. [MS-WF-01].
"""

TDD_SYSTEM = """
You are the Technical Design Agent of a Dynamics 365 F&O delivery team.
Write the Technical Design Document (TDD) for the approved specification.

- Follow the company design and development standards: solution overview, object list (table and form
  extensions, Chain of Command classes, SysOperation batch classes, security privileges and duties),
  logic, data model changes, error handling and logging, performance, deployment.
- Extensions only (no overlayering), CUS prefix, configurable values in a parameter table.
- Section IDs TDD-1, TDD-2, ... Each section lists the requirement IDs it covers, and every requirement
  must appear in at least one section.
- Use only the immutable facts. Cite evidence IDs in the text, e.g. [DES-2025-031].
"""

TEST_SYSTEM = """
You are the Test Agent of a Dynamics 365 F&O delivery team.
Create test cases for the approved specification, IDs TC-01, TC-02, ...

- Types: Functional, Technical, Integration and Regression.
- Every requirement ID needs at least one test case.
- Threshold rules need boundary tests at, just below and just above the limit.
- Security rules need a negative test proving an unauthorized user is blocked.
- Add regression tests for standard processes that must keep working.
- Use the exact values from the immutable facts.
"""

VALIDATION_SYSTEM = """
You are the Validation Agent of a Dynamics 365 F&O delivery team.
Compare the artefacts with the immutable facts and with each other.

Report only real problems:
- a wrong value (amount, currency, legal entity, role, number of days, hold type, deadline),
- a contradiction between the design, the acceptance criteria and the tests,
- behaviour that is not in the approved requirements.
For each issue name the responsible agent: planning, functional_design, technical_design or tests.
If everything is consistent, return an empty list of issues.
"""

REVISION_INSTRUCTIONS = """
Revise your previous output. It failed validation.

Validation feedback:
{feedback}

Lessons from earlier attempts (reflexion memory):
{reflections}

Return the complete corrected artefact. Keep everything that was already correct.
"""

## Layer 1: Deterministic Traceability

`build_traceability` maps every approved requirement ID to the artefacts that reference it.
`traceability_issues` then reports:

- a requirement without an acceptance criterion (Planning Agent),
- a requirement without a design section (Design Agent),
- a requirement without a test (Test Agent),
- any reference to an ID that is not an approved requirement (the agent that produced it).

In [ ]:
COLUMN_OWNER = {
    "acceptance_criteria": "planning",
    "tasks": "planning",
    "fdd_sections": "functional_design",
    "tdd_sections": "technical_design",
    "tests": "tests",
}


def build_traceability(requirements: list, plan: dict, fdd: dict | None, tdd: dict | None, tests: dict):
    """Map every requirement ID to the acceptance criteria, tasks, design sections and tests that cover it."""
    matrix = {r["req_id"]: {column: [] for column in COLUMN_OWNER} for r in requirements}
    unknown_refs = []

    def add(req_ids, column, ref):
        for req_id in req_ids:
            if req_id in matrix:
                matrix[req_id][column].append(ref)
            else:
                unknown_refs.append((req_id, column, ref))

    for story in plan["user_stories"]:
        for ac in story["acceptance_criteria"]:
            add(ac["req_ids"], "acceptance_criteria", ac["ac_id"])
        for task in story["tasks"]:
            add(task["req_ids"], "tasks", task["task_id"])
    for design, column in ((fdd, "fdd_sections"), (tdd, "tdd_sections")):
        for section in (design or {}).get("sections", []):
            add(section["req_ids"], column, section["section_id"])
    for test in tests["test_cases"]:
        add(test["req_ids"], "tests", test["test_id"])

    return matrix, unknown_refs


def traceability_issues(matrix: dict, unknown_refs: list, design_documents: str) -> list:
    issues = []
    design_owner = "functional_design" if "FDD" in design_documents else "technical_design"

    for req_id, row in matrix.items():
        if not row["acceptance_criteria"]:
            issues.append({"severity": "Major", "responsible_agent": "planning", "req_ids": [req_id],
                           "description": f"{req_id} has no acceptance criterion.",
                           "fix_instruction": f"Add a Given/When/Then acceptance criterion for {req_id}."})
        if not row["fdd_sections"] and not row["tdd_sections"]:
            issues.append({"severity": "Major", "responsible_agent": design_owner, "req_ids": [req_id],
                           "description": f"{req_id} is not covered by any design section.",
                           "fix_instruction": f"Add or extend a design section that covers {req_id}."})
        if not row["tests"]:
            issues.append({"severity": "Major", "responsible_agent": "tests", "req_ids": [req_id],
                           "description": f"{req_id} has no test case.",
                           "fix_instruction": f"Add at least one test case for {req_id}."})

    for req_id, column, ref in unknown_refs:
        issues.append({"severity": "Major", "responsible_agent": COLUMN_OWNER[column], "req_ids": [req_id],
                       "description": f"{ref} references {req_id}, which is not an approved requirement.",
                       "fix_instruction": f"Only reference approved requirement IDs: {', '.join(matrix)}."})
    return issues


def print_matrix(matrix: dict) -> None:
    print(f"{'Requirement':<12}{'Accept. criteria':<22}{'FDD':<16}{'TDD':<16}{'Tests'}")
    for req_id, row in matrix.items():
        print(f"{req_id:<12}{', '.join(row['acceptance_criteria']) or '-':<22}"
              f"{', '.join(row['fdd_sections']) or '-':<16}{', '.join(row['tdd_sections']) or '-':<16}"
              f"{', '.join(row['tests']) or 'MISSING'}")
matrix, unknown_refs = build_traceability(APPROVED_REQUIREMENTS, SAMPLE_PLAN, DRAFT_FDD, DRAFT_TDD, DRAFT_TESTS)
print_matrix(matrix)

show("TRACEABILITY ISSUES", traceability_issues(matrix, unknown_refs, CLASSIFICATION["design_documents"]))

## Layer 2: LLM Consistency Review

The traceability check cannot see that FDD-1 says **5,000 USD**, because the section still references the
right requirement. The LLM review receives the immutable facts and a compact digest of all artefacts, and
looks for contradictions of this kind.

In [ ]:
def artefact_digest(state: dict) -> str:
    """Compact view of all artefacts for the LLM consistency review."""
    lines = ["IMMUTABLE FACTS:", *state["locked_facts"], "", "ACCEPTANCE CRITERIA:"]
    for story in state["plan"]["user_stories"]:
        for ac in story["acceptance_criteria"]:
            lines.append(f"{ac['ac_id']} ({', '.join(ac['req_ids'])}): Given {ac['given']}, "
                         f"when {ac['when']}, then {ac['then']}")
    for key in ("fdd", "tdd"):
        design = state.get(key)
        if design:
            lines += ["", f"{design['doc_type']} SECTIONS:"]
            lines += [f"{s['section_id']} {s['heading']} ({', '.join(s['req_ids'])}): {s['content'][:600]}"
                      for s in design["sections"]]
    lines += ["", "TEST CASES:"]
    lines += [f"{t['test_id']} [{t['test_type']}] ({', '.join(t['req_ids'])}) {t['title']}: "
              f"expected {t['expected_result']}" for t in state["tests"]["test_cases"]]
    return "\n".join(lines)


@traceable(name="Validation Agent")
def validate_package(state: dict):
    """Validation Agent: deterministic traceability checks + LLM consistency review."""
    matrix, unknown_refs = build_traceability(
        state["requirements"], state["plan"], state.get("fdd"), state.get("tdd"), state["tests"]
    )
    issues = traceability_issues(matrix, unknown_refs, state["classification"]["design_documents"])

    review = ask_structured(ConsistencyReview, VALIDATION_SYSTEM, artefact_digest(state))
    issues += [issue.model_dump() for issue in review.issues]

    for number, issue in enumerate(issues, start=1):
        issue["issue_id"] = f"V-{number:02d}"
    return matrix, issues


AGENT_SPECS = {
    # agent name: (system prompt, output schema, state key)
    "planning": (PLANNING_SYSTEM, DeliveryPlan, "plan"),
    "functional_design": (FDD_SYSTEM, DesignDocument, "fdd"),
    "technical_design": (TDD_SYSTEM, DesignDocument, "tdd"),
    "tests": (TEST_SYSTEM, TestSuite, "tests"),
}


@traceable(name="Reflexion revision")
def revise_artefact(agent: str, state: dict, issues: list) -> dict:
    """Send the artefact back to the responsible agent with the validation feedback and reflexion memory."""
    system_prompt, schema, key = AGENT_SPECS[agent]
    feedback = "\n".join(
        f"- {i['issue_id']} [{i['severity']}] {i['description']} Fix: {i['fix_instruction']}"
        for i in issues if i["responsible_agent"] == agent
    )
    user_prompt = (
        f"APPROVED PACKAGE:\n{handoff_package(state)}\n\n"
        f"YOUR PREVIOUS OUTPUT:\n{json.dumps(state[key], indent=2, ensure_ascii=False)}\n\n"
        + REVISION_INSTRUCTIONS.format(feedback=feedback, reflections="\n".join(state.get("reflections", [])) or "none")
    )
    return ask_structured(schema, system_prompt, user_prompt).model_dump()


def is_blocking(issue: dict) -> bool:
    return issue["severity"] in ("Blocker", "Major")
state = {
    "request_id": REQUEST_ID,
    "source_text": CUSTOMER_REQUEST,
    "requirements": APPROVED_REQUIREMENTS,
    "classification": CLASSIFICATION,
    "evidence": EVIDENCE,
    "approval_status": {"specification": "approved"},
    "locked_facts": LOCKED_FACTS,
    "plan": SAMPLE_PLAN,
    "fdd": DRAFT_FDD,
    "tdd": DRAFT_TDD,
    "tests": DRAFT_TESTS,
}

matrix, issues = validate_package(state)
for issue in issues:
    print(f"{issue['issue_id']} [{issue['severity']}] -> {issue['responsible_agent']}: {issue['description']}")

## The Reflexion Loop in LangGraph

**validate -> (blocking issues?) -> revise -> validate -> ... -> done**

- `validate` runs both layers.
- The router ends the loop when no Blocker or Major issue remains, or when `MAX_REVISIONS` is reached. In
  that case the remaining issues are shown to the human at final approval instead of being hidden.
- `revise` sends each artefact with blocking issues back to its responsible agent. Before that, every
  failure is written to the **reflexion memory**, so the agent sees what went wrong in earlier attempts and
  does not repeat it.

In [ ]:
MAX_REVISIONS = 3


class ReviewState(TypedDict, total=False):
    request_id: str
    source_text: str
    requirements: list
    classification: dict
    evidence: list
    approval_status: dict
    locked_facts: list
    plan: dict
    fdd: dict | None
    tdd: dict | None
    tests: dict
    matrix: dict
    issues: list
    revision_count: int
    reflections: Annotated[list, operator.add]


def validate_node(state: ReviewState) -> dict:
    matrix, issues = validate_package(state)
    blocking = [i for i in issues if is_blocking(i)]
    print(f"[validate] round {state.get('revision_count', 0)}: {len(issues)} issues, {len(blocking)} blocking")
    return {"matrix": matrix, "issues": issues}


def route_after_validation(state: ReviewState) -> str:
    if not any(is_blocking(i) for i in state["issues"]):
        return "done"
    if state.get("revision_count", 0) >= MAX_REVISIONS:
        print("[validate] maximum revisions reached - remaining issues go to the human reviewer")
        return "done"
    return "revise"


def revise_node(state: ReviewState) -> dict:
    blocking = [i for i in state["issues"] if is_blocking(i)]
    attempt = state.get("revision_count", 0) + 1

    # Reflexion memory: record why this attempt failed before asking for a revision.
    lessons = [f"Attempt {attempt}: {i['responsible_agent']} - {i['description']} -> {i['fix_instruction']}"
               for i in blocking]
    memory = {**state, "reflections": state.get("reflections", []) + lessons}

    updates = {"revision_count": attempt, "reflections": lessons}
    for agent in dict.fromkeys(i["responsible_agent"] for i in blocking):  # unique, in order
        key = AGENT_SPECS[agent][2]
        print(f"[revise] sending '{key}' back to the {agent} agent")
        updates[key] = revise_artefact(agent, memory, blocking)
    return updates


graph = StateGraph(ReviewState)
graph.add_node("validate", validate_node)
graph.add_node("revise", revise_node)
graph.add_edge(START, "validate")
graph.add_conditional_edges("validate", route_after_validation, {"revise": "revise", "done": END})
graph.add_edge("revise", "validate")

review_loop = graph.compile()
print(review_loop.get_graph().draw_mermaid())

## Run the Loop

In [ ]:
final = review_loop.invoke({**state, "revision_count": 0})

print("\n========== FINAL TRACEABILITY MATRIX ==========\n")
print_matrix(final["matrix"])

show("REMAINING ISSUES", [f"{i['issue_id']} [{i['severity']}] {i['description']}" for i in final["issues"]] or "none")
show("REFLEXION MEMORY", final["reflections"])

In [ ]:
print("FDD-1 before:", DRAFT_FDD["sections"][0]["content"])
print("FDD-1 after: ", final["fdd"]["sections"][0]["content"])
print()
print("Tests for REQ-005 after revision:",
      [t["test_id"] for t in final["tests"]["test_cases"] if "REQ-005" in t["req_ids"]])
print("Unknown IDs still referenced:",
      sorted({r for t in final["tests"]["test_cases"] for r in t["req_ids"]} - {r["req_id"] for r in APPROVED_REQUIREMENTS}) or "none")

## Observation

The two layers found different problems. The deterministic check found that REQ-005 had no test and that
TC-06 referenced REQ-009, which was never approved. Neither needs an LLM, and code finds them every time.
The LLM review found what the matrix cannot see: FDD-1 still pointed to the correct requirement, but it used
5,000 USD instead of the approved 10,000 USD.

Each issue named its responsible agent, so the loop sent the FDD back to the Functional Design Agent and the
tests back to the Test Agent. The plan and the TDD were not touched. The reflexion memory recorded why each
attempt failed, and the revised artefacts were validated again until no blocking issue remained, within
`MAX_REVISIONS`.

The loop never approves anything by itself. In the final notebook, the validated package still stops at the
human final approval, where any remaining minor issues and the full traceability matrix are shown.